In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col, desc
from google.colab import files

In [2]:
spark = SparkSession.builder \
    .appName("ProductsSoldAnalysis") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [3]:
uploaded = files.upload()

file_path = "/content/OnlineRetail.csv"

Saving OnlineRetail.csv to OnlineRetail.csv


In [4]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("encoding", "ISO-8859-1") \
    .csv(file_path)

print("Dataset Loaded Successfully!")

df.show(5)
df.printSchema()

Dataset Loaded Successfully!
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows
root
 |-- InvoiceNo:

In [5]:
cleaned_df = df.filter(
    col("Description").isNotNull()
).filter(
    ~col("InvoiceNo").cast("string").startswith("C")
).filter(
    col("Quantity") > 0
)

print("Data Preprocessing Completed!")

print("Total Records:", cleaned_df.count())

Data Preprocessing Completed!
Total Records: 530693


In [6]:
product_sales = cleaned_df.groupBy("Description") \
    .agg(
        sum("Quantity").alias("Total_Products_Sold")
    )

print("Total Products Sold in Each Product Group:")
product_sales.show(20, truncate=False)

Total Products Sold in Each Product Group:
+----------------------------------+-------------------+
|Description                       |Total_Products_Sold|
+----------------------------------+-------------------+
|10 COLOUR SPACEBOY PEN            |6564               |
|SET/10 BLUE POLKADOT PARTY CANDLES|4015               |
|POTTING SHED SOW 'N' GROW SET     |2                  |
|PAPERWEIGHT KINGS CHOICE          |168                |
|WOVEN BERRIES CUSHION COVER       |162                |
|WHITE/PINK MINI CRYSTALS NECKLACE |10                 |
|SET/3 RED GINGHAM ROSE STORAGE BOX|3040               |
|MAGNETS PACK OF 4 VINTAGE LABELS  |1323               |
|WHITE CHRYSANTHEMUMS ART FLOWER   |55                 |
|WHITE FRANGIPANI NECKLACE         |39                 |
|SILVER FABRIC MIRROR              |458                |
|PINK  HONEYCOMB PAPER FAN         |499                |
|PINK BOUDOIR T-LIGHT HOLDER       |12                 |
|GLASS CAKE COVER AND PLATE        |2        

In [7]:
sorted_sales = product_sales.orderBy(
    desc("Total_Products_Sold")
)

print("Products Sorted by Sales:")
sorted_sales.show(20, truncate=False)

Products Sorted by Sales:
+----------------------------------+-------------------+
|Description                       |Total_Products_Sold|
+----------------------------------+-------------------+
|PAPER CRAFT , LITTLE BIRDIE       |80995              |
|MEDIUM CERAMIC TOP STORAGE JAR    |78033              |
|WORLD WAR 2 GLIDERS ASSTD DESIGNS |55047              |
|JUMBO BAG RED RETROSPOT           |48478              |
|WHITE HANGING HEART T-LIGHT HOLDER|37895              |
|POPCORN HOLDER                    |36761              |
|ASSORTED COLOUR BIRD ORNAMENT     |36461              |
|PACK OF 72 RETROSPOT CAKE CASES   |36419              |
|RABBIT NIGHT LIGHT                |30788              |
|MINI PAINT SET VINTAGE            |26633              |
|PACK OF 12 LONDON TISSUES         |26339              |
|PACK OF 60 PINK PAISLEY CAKE CASES|24854              |
|VICTORIAN GLASS HANGING T-LIGHT   |24304              |
|ASSORTED COLOURS SILK FAN         |23826              |
|BROC

In [8]:
total_products = cleaned_df.agg(
    sum("Quantity").alias("Total_Quantity_Sold")
)

print("Overall Product Sales:")
total_products.show()

Overall Product Sales:
+-------------------+
|Total_Quantity_Sold|
+-------------------+
|            5628434|
+-------------------+



In [9]:
spark.stop()

print("Spark Session Stopped Successfully!")

Spark Session Stopped Successfully!
